In [9]:
import sys
import os
import importlib

project_root = os.path.abspath(os.path.join('..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import scripts.analisis_financiero_flota
importlib.reload(scripts.analisis_financiero_flota)
from scripts.analisis_financiero_flota import FleetFinancialModel

print("✅ Motor financiero recargado. Parámetros interactivos activos.")

✅ Motor financiero recargado. Parámetros interactivos activos.


# 📈 Análisis Financiero: Compra vs. Leasing vs. Renting (51 Camiones 44t)

Este informe técnico evalúa las tres principales alternativas de adquisición de una flota pesada para un horizonte de **5 años (60 meses)**, bajo la normativa fiscal española vigente (IS 25%).

## ⚙️ 1. Consola de Parámetros
Modifica los valores de abajo para ajustar el escenario. Todos los números grandes usan el separador `_` por legibilidad.

In [10]:
# === BLOQUE 1: INVERSIÓN Y EQUIPO ===
N_CAMIONES = 51
PRECIO_UNITARIO = 140_000        # Euros por unidad (44t)
HORIZONTE_AÑOS = 5               # Años de operación

# === BLOQUE 2: COSTES OPERATIVOS (OPEX ANUAL) ===
MANTENIMIENTO = 8_500.0          # €/año/unidad
SEGUROS = 3_200.0                # €/año/unidad
NEUMATICOS = 2_400.0             # €/año/unidad
ADMIN_COMPRA = 600.0             # Gestión en propia
ADMIN_LEASING = 300.0            # Gestión en leasing
CUOTA_RENTING_MES = 2_800.0      # Cuota all-inclusive

# === BLOQUE 3: TASAS Y FISCALIDAD ===
WACC = 0.09                      # Tasa de descuento (9%)
IMPUESTO_SOCIEDADES = 0.25       # IS España (25%)
TAE_PRESTAMO = 0.055             # Interés Compra (5.5%)
TAE_LEASING = 0.048              # Interés Leasing (4.8%)
VALOR_RESIDUAL_PCT = 0.35        # Valor mercado al año 5 (35%)

print("✅ Parámetros cargados correctamente.")

✅ Parámetros cargados correctamente.


## 2. Metodología de Valoración Financiera

### A. Valor Actual Neto (VAN / NPV)
Representa el valor neto a día de hoy de todos los flujos de caja futuros (entradas y salidas), descontados a una tasa de riesgo proyectada ($k$ o WACC):

$$\text{VAN} = \sum_{t=0}^{n} \frac{CF_t}{(1 + k)^t}$$

### B. Tasa Interna de Retorno (TIR / IRR)
Es la tasa de rentabilidad intrínseca del proyecto que iguala el VAN a cero:

$$0 = \sum_{t=0}^{n} \frac{CF_t}{(1 + \text{TIR})^t}$$

### C. Cálculo de la Anualidad Francesa
Para estimar el servicio de la deuda mensual (amortización francesa), utilizamos la fórmula:

$$A = P \cdot \frac{i(1+i)^n}{(1+i)^n - 1}$$

### D. Escudo Fiscal Acelerado (Art. 106 LIS)
$$\text{Escudo Fiscal Leasing} = (\text{Intereses} + \text{Principal Deducible} + \text{OpEx}) \cdot \text{IS}$$
*(Donde el Principal Deducible tiene un límite del doble de la amortización lineal máxima)*.

In [11]:
model = FleetFinancialModel(
    n_trucks=N_CAMIONES,
    unit_price=PRECIO_UNITARIO,
    horizon_years=HORIZONTE_AÑOS,
    wacc=WACC,
    is_rate=IMPUESTO_SOCIEDADES,
    maint_year=MANTENIMIENTO,
    ins_year=SEGUROS,
    tires_year=NEUMATICOS,
    admin_purchase=ADMIN_COMPRA,
    admin_leasing=ADMIN_LEASING,
    renting_fee=CUOTA_RENTING_MES,
    loan_tae=TAE_PRESTAMO,
    lease_tae=TAE_LEASING,
    residual_val_pct=VALOR_RESIDUAL_PCT
)

summary = model.get_summary_table()
summary.style.format({"TCO Flota 5A (Bruto)": "{0:,.2f} €", "Ahorro Fiscal Acum.": "{0:,.2f} €", "Coste Neto Flota (TCO-Fiscal)": "{0:,.2f} €", "VAN Project (WACC 9%)": "{0:,.2f} €"})


,Escenario,TCO Flota 5A (Bruto),Ahorro Fiscal Acum.,Coste Neto Flota (TCO-Fiscal),Coste Mensual x Unidad,VAN Project (WACC 9%),TIR (%)
0,Compra Financiada,"9,491,600.30 €","2,573,712.57 €","6,917,887.72 €",3101.830163,"-5,973,905.17 €",-70.285534
1,Leasing Financiero,"9,752,467.33 €","2,728,179.33 €","7,024,288.00 €",3187.080827,"-5,862,865.81 €",-87.334911
2,Renting Operativo,"8,568,000.00 €","2,142,000.00 €","6,426,000.00 €",2800.000000,"-4,998,979.80 €",0.000000


In [12]:
print("### 📊 ANÁLISIS DE SENSIBILIDAD VAN (Millones €) ###")
print("Interpretación: Cómo afecta el Precio del Camión y su Valor Residual al VAN Final.")
display(model.sensitivity_table())

### 📊 ANÁLISIS DE SENSIBILIDAD VAN (Millones €) ###
Interpretación: Cómo afecta el Precio del Camión y su Valor Residual al VAN Final.


,Residual 25%,Residual 35%,Residual 45%
Precio -10%,-5.80M,-5.49M,-5.18M
Precio Base,-6.21M,-5.86M,-5.51M
Precio +10%,-6.62M,-6.23M,-5.85M


## 📚 Bibliografía y Referencias Normativas

1. **Gisbert, J. (2020)**. *Gestión de flotas y costes de transporte*. Editorial Logis. (Normas TCO).
2. **Damodaran, A. (2014)**. *Applied Corporate Finance*. Wiley. (Metodología NPV/IRR).
3. **Ley 27/2014 (LIS)**. Artículo 106: Régimen especial de contratos de arrendamiento financiero en España.
4. **Real Decreto 634/2015**. Reglamento del Impuesto sobre Sociedades. Tablas de amortización lineal oficiales.
5. **Brigham, E. F., & Ehrhardt, M. C. (2016)**. *Financial Management: Theory & Practice*. Cengage.
6. **MITMA (2026)**. *Observatorio de Costes*. Metodología de cálculo de explotación para vehículos de 44t.